# Safe Driver Prediction: 深度学习与经典模型系统评估

> 任务目标：对 TabNet、FT-Transformer 与经典模型（XGBoost、LightGBM、CatBoost、Logistic Regression、Random Forest）进行严格、可复现实验对比。

## 实验协议（严格执行）
- 固定数据划分：Train/Valid/Test = 60%/20%/20%，且所有方法共享同一划分
- 随机种子：至少 3 个种子，报告均值与标准差
- 调参预算：每个方法 10 个 Optuna trials（可统一通过配置变量管理）
- 分类指标：Accuracy、AUC-ROC、F1-score
- 实用性评估：预处理复杂度、训练成本、推理延迟、模型体积、稳定性、可解释性

## Notebook 阶段
1. 数据清洗与预处理
2. 特征工程
3. 模型训练与调参（每个模型独立代码块）
4. 模型预测与综合分析

In [1]:
# ===== 0) 环境准备与全局配置 =====
import os
import gc
import time
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.INFO)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# 可复现实验配置
FAST_MODE = True

if FAST_MODE:
    # 快速模式：用于先验证流程，显著缩短训练时间
    SEEDS = [123]
    N_TRIALS = 1
    MAX_TRAIN_ROWS = 80000
    MAX_VALID_ROWS = 30000
    MAX_TRAIN_ROWS_FT = 15000
    MAX_VALID_ROWS_FT = 6000
    TABNET_MAX_EPOCHS = 40
    FT_MAX_EPOCHS = 8
    INFER_REPEAT = 1
    FT_FORCE_CPU = True
else:
    # 正式模式：用于最终报告
    SEEDS = [123, 456, 789]
    N_TRIALS = 10
    MAX_TRAIN_ROWS = None
    MAX_VALID_ROWS = None
    MAX_TRAIN_ROWS_FT = None
    MAX_VALID_ROWS_FT = None
    TABNET_MAX_EPOCHS = 100
    FT_MAX_EPOCHS = 20
    INFER_REPEAT = 5
    FT_FORCE_CPU = False

VALID_SIZE = 0.20  # 全量中的 20%
TEST_SIZE = 0.20   # 全量中的 20%
TARGET_COL = "target"
DATA_PATH = Path("dataset.csv")
RESULT_DIR = Path("artifacts")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Config loaded")
print(f"[INFO] FAST_MODE: {FAST_MODE}")
print(f"[INFO] Seeds: {SEEDS}")
print(f"[INFO] Optuna trials per method: {N_TRIALS}")
print(f"[INFO] Train/Valid cap for tuning: {MAX_TRAIN_ROWS}/{MAX_VALID_ROWS}")
print(f"[INFO] FT Train/Valid cap: {MAX_TRAIN_ROWS_FT}/{MAX_VALID_ROWS_FT}")
print(f"[INFO] FT_FORCE_CPU: {FT_FORCE_CPU}")

[INFO] Config loaded
[INFO] FAST_MODE: True
[INFO] Seeds: [123]
[INFO] Optuna trials per method: 1
[INFO] Train/Valid cap for tuning: 80000/30000
[INFO] FT Train/Valid cap: 15000/6000
[INFO] FT_FORCE_CPU: True


c:\Users\27335\Desktop\DSS5104\Assign2\safe_driver_prediction_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 设置最大显示行数
pd.set_option('display.max_rows', None)
# 设置最大显示列数
pd.set_option('display.max_columns', None)
# 设置列宽，确保内容不被截断
pd.set_option('display.max_colwidth', None)
# 设置显示的浮点数精度
pd.set_option('display.precision', 2)

In [3]:
import importlib
import subprocess
import sys
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

print("[INFO] Core model libraries imported successfully")

[INFO] Core model libraries imported successfully


## 1) 数据清洗与预处理

本阶段目标：
- 正确处理缺失值标记（`-1`）
- 自动识别特征类型（`*_bin`、`*_cat`、其余数值/有序）
- 固定并保存 train/valid/test 切分索引，保证所有模型使用同一划分
- 输出数据质量与缺失情况日志

In [4]:
# ===== 1.1 读取数据与基础审计 =====
assert DATA_PATH.exists(), f"dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(f"[INFO] Loaded data shape: {df.shape}")
print(f"[INFO] Columns: {len(df.columns)}")

assert TARGET_COL in df.columns, f"target column '{TARGET_COL}' not found"
print("[INFO] target distribution:")
print(df[TARGET_COL].value_counts(normalize=True).rename("ratio"))

# 将 -1 视为缺失（题目约定）
feature_cols = [c for c in df.columns if c != TARGET_COL]
df[feature_cols] = df[feature_cols].replace(-1, np.nan)

missing_stats = df[feature_cols].isna().mean().sort_values(ascending=False)
print("[INFO] Top-20 missing ratio columns:")
print(missing_stats.head(20))

print("[INFO] Sample rows:")
display(df.head(5))

[INFO] Loaded data shape: (595212, 59)
[INFO] Columns: 59
[INFO] target distribution:
target
0    0.96
1    0.04
Name: ratio, dtype: float64
[INFO] Top-20 missing ratio columns:
ps_car_03_cat    6.91e-01
ps_car_05_cat    4.48e-01
ps_reg_03        1.81e-01
ps_car_14        7.16e-02
ps_car_07_cat    1.93e-02
ps_ind_05_cat    9.76e-03
ps_car_09_cat    9.56e-04
ps_ind_02_cat    3.63e-04
ps_car_01_cat    1.80e-04
ps_ind_04_cat    1.39e-04
ps_car_02_cat    8.40e-06
ps_car_11        8.40e-06
ps_car_12        1.68e-06
ps_ind_08_bin    0.00e+00
ps_ind_09_bin    0.00e+00
ps_ind_11_bin    0.00e+00
ps_ind_12_bin    0.00e+00
ps_ind_13_bin    0.00e+00
ps_ind_17_bin    0.00e+00
ps_ind_16_bin    0.00e+00
dtype: float64
[INFO] Sample rows:


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2.0,5,1.0,0.0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.72,10.0,1.0,NaN,0,1.0,4,1.0,0,0.0,1,12,2.0,0.40,0.88,0.37,3.61,0.6,0.5,0.2,3,1,10,1,10,1,5,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1.0,7,0.0,0.0,0,0,1,0,0,0,0,0,0,3,0,0,1,0.8,0.4,0.77,11.0,1.0,NaN,0,NaN,11,1.0,1,2.0,1,19,3.0,0.32,0.62,0.39,2.45,0.3,0.1,0.3,2,1,9,5,8,1,7,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4.0,9,1.0,0.0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,NaN,7.0,1.0,NaN,0,NaN,14,1.0,1,2.0,1,60,1.0,0.32,0.64,0.35,3.32,0.5,0.7,0.1,2,2,9,1,8,2,7,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1.0,2,0.0,0.0,1,0,0,0,0,0,0,0,0,8,1,0,0,0.9,0.2,0.58,7.0,1.0,0.0,0,1.0,11,1.0,1,3.0,1,104,1.0,0.37,0.54,0.29,2.00,0.6,0.9,0.1,2,4,7,1,8,4,2,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2.0,0,1.0,0.0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.84,11.0,1.0,NaN,0,NaN,14,1.0,1,2.0,1,82,3.0,0.32,0.57,0.37,2.00,0.4,0.6,0.0,2,2,6,3,10,2,12,3,1,1,3,0,0,0,1,1,0


In [5]:
# ===== 1.2 自动识别特征类型 =====
all_features = [c for c in df.columns if c != TARGET_COL]

bin_cols = [c for c in all_features if c.endswith("_bin")]
cat_cols = [c for c in all_features if c.endswith("_cat")]
num_cols = [c for c in all_features if c not in set(bin_cols + cat_cols)]

# 按题目提示统计前缀组（ind/reg/car/calc）
def get_prefix(col):
    parts = col.split("_")
    return parts[1] if len(parts) > 1 else "unknown"

prefix_group = pd.Series([get_prefix(c) for c in all_features]).value_counts()

print(f"[INFO] Binary columns: {len(bin_cols)}")
print(f"[INFO] Categorical columns: {len(cat_cols)}")
print(f"[INFO] Numeric/Ordinal columns: {len(num_cols)}")
print("[INFO] Prefix group counts:")
print(prefix_group)

# 高基数类别特征检查（用于分析预处理难度）
high_cardinality = []
for c in cat_cols:
    n_unique = df[c].nunique(dropna=True)
    if n_unique >= 20:
        high_cardinality.append((c, n_unique))
high_cardinality = sorted(high_cardinality, key=lambda x: x[1], reverse=True)

print(f"[INFO] High-cardinality categorical features (>=20 unique): {len(high_cardinality)}")
print(high_cardinality[:20])

[INFO] Binary columns: 17
[INFO] Categorical columns: 14
[INFO] Numeric/Ordinal columns: 27
[INFO] Prefix group counts:
calc       20
ind        18
car        16
reg         3
unknown     1
Name: count, dtype: int64
[INFO] High-cardinality categorical features (>=20 unique): 1
[('ps_car_11_cat', 104)]


In [6]:
# ===== 1.3 固定 60/20/20 切分（分层） =====
X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].astype(int).copy()

# 第一步：先切出 test=20%
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEEDS[0], stratify=y
)

# 第二步：再从剩余 80% 中切出 valid=20%（即总量 16%）
valid_ratio_in_train_valid = VALID_SIZE / (1 - TEST_SIZE)  # 0.2/0.8 = 0.25
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=valid_ratio_in_train_valid,
    random_state=SEEDS[0],
    stratify=y_train_valid
)

print("[INFO] Split summary:")
print(f"Train: {X_train.shape}, ratio={len(X_train)/len(X):.3f}")
print(f"Valid: {X_valid.shape}, ratio={len(X_valid)/len(X):.3f}")
print(f"Test : {X_test.shape}, ratio={len(X_test)/len(X):.3f}")

# 保存索引，保证完全可复现
split_idx = {
    "train_index": X_train.index.tolist(),
    "valid_index": X_valid.index.tolist(),
    "test_index": X_test.index.tolist(),
}
with open(RESULT_DIR / "fixed_split_indices.json", "w", encoding="utf-8") as f:
    json.dump(split_idx, f, ensure_ascii=False, indent=2)
print(f"[INFO] Saved split indices to {RESULT_DIR / 'fixed_split_indices.json'}")

[INFO] Split summary:
Train: (357126, 58), ratio=0.600
Valid: (119043, 58), ratio=0.200
Test : (119043, 58), ratio=0.200
[INFO] Saved split indices to artifacts\fixed_split_indices.json


## 2) 特征工程与统一工具函数

本阶段目标：
- 为不同模型族构造可复用的预处理 pipeline
- 提供统一评估、延迟测试、模型体积统计、结果汇总函数
- 为后续每个模型的独立训练代码块减少重复实现

In [7]:
# ===== 2.1 统一预处理器与评估工具 =====
def build_sklearn_preprocessor(model_family: str):
    """
    model_family: 'linear' | 'tree'
    linear: 数值标准化 + 类别 one-hot
    tree  : 数值中位数填充 + 类别 ordinal 编码（便于树模型处理）
    """
    if model_family == "linear":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ])
    elif model_family == "tree":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
    else:
        raise ValueError(f"unknown model_family={model_family}")

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols + bin_cols),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop")
    return preprocessor


def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def benchmark_inference(model_predict_proba_fn, X_input, n_repeat=INFER_REPEAT, batch_size=2048):
    # 记录每样本推理耗时（毫秒）
    n = len(X_input)
    idx = np.arange(n)
    latencies = []
    for _ in range(n_repeat):
        np.random.shuffle(idx)
        t0 = time.perf_counter()
        for i in range(0, n, batch_size):
            batch_idx = idx[i:i+batch_size]
            _ = model_predict_proba_fn(X_input.iloc[batch_idx])
        t1 = time.perf_counter()
        elapsed = (t1 - t0) / n * 1000.0
        latencies.append(elapsed)
    return float(np.mean(latencies)), float(np.std(latencies))


def estimate_model_size_mb(model_obj):
    # 通过临时序列化估计模型体积
    import tempfile
    import joblib
    with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tmp:
        joblib.dump(model_obj, tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
    try:
        os.remove(tmp.name)
    except Exception:
        pass
    return float(size_mb)


def print_seed_metrics(model_name, seed_results):
    print(f"\n[SUMMARY] {model_name}")
    for m in ["accuracy", "auc_roc", "f1", "train_seconds", "infer_ms_per_sample", "model_size_mb"]:
        vals = [r[m] for r in seed_results]
        print(f"  - {m:20s}: mean={np.mean(vals):.6f}, std={np.std(vals):.6f}")

In [8]:
# ===== 2.2 结果容器与日志辅助 =====
all_results = {}
best_models = {}
best_preprocessors = {}

def optuna_log_callback(study, trial):
    print(
        f"[Optuna] Trial {trial.number:03d} | value={trial.value:.6f} | "
        f"params={trial.params}"
    )

print("[INFO] Utilities ready")

[INFO] Utilities ready


## 3) 模型训练与调参（每个模型独立代码块）

说明：
- 每个模型都按相同预算执行 Optuna 调参（`N_TRIALS`）
- 每个模型在相同数据划分上，使用 3 个随机种子重复训练
- 输出：每个 seed 的验证日志、测试集指标、训练耗时、推理耗时、模型大小

In [9]:
# ===== 2.3 经典模型统一调参与评估执行器 =====
def run_classical_experiment(
    model_name,
    model_builder_fn,
    param_suggester_fn,
    model_family,
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    ):
    seed_results = []
    best_auc = -np.inf
    best_pack = None

    for seed in seeds:
        print(f"\n{'='*20} {model_name} | seed={seed} {'='*20}")
        np.random.seed(seed)
        random.seed(seed)

        # FAST_MODE: 对调参阶段使用子样本，显著缩短训练时间
        X_train_seed = X_train
        y_train_seed = y_train
        X_valid_seed = X_valid
        y_valid_seed = y_valid

        if MAX_TRAIN_ROWS is not None and len(X_train_seed) > MAX_TRAIN_ROWS:
            idx = X_train_seed.sample(n=MAX_TRAIN_ROWS, random_state=seed).index
            X_train_seed = X_train_seed.loc[idx]
            y_train_seed = y_train_seed.loc[idx]

        if MAX_VALID_ROWS is not None and len(X_valid_seed) > MAX_VALID_ROWS:
            idx = X_valid_seed.sample(n=MAX_VALID_ROWS, random_state=seed).index
            X_valid_seed = X_valid_seed.loc[idx]
            y_valid_seed = y_valid_seed.loc[idx]

        if use_native_cat:
            # CatBoost 原生类别支持路径
            Xtr = X_train_seed.copy()
            Xva = X_valid_seed.copy()
            Xte = X_test.copy()

            for c in cat_cols:
                Xtr[c] = Xtr[c].astype("string").fillna("MISSING")
                Xva[c] = Xva[c].astype("string").fillna("MISSING")
                Xte[c] = Xte[c].astype("string").fillna("MISSING")
        else:
            preprocessor = build_sklearn_preprocessor(model_family)
            Xtr = preprocessor.fit_transform(X_train_seed)
            Xva = preprocessor.transform(X_valid_seed)
            Xte = preprocessor.transform(X_test)

        def objective(trial):
            params = param_suggester_fn(trial)
            model = model_builder_fn(params, seed)

            t0 = time.perf_counter()
            if use_native_cat:
                model.fit(
                    Xtr, y_train_seed,
                    eval_set=(Xva, y_valid_seed),
                    cat_features=cat_cols,
                    use_best_model=True,
                    verbose=100
                )
                yva_prob = model.predict_proba(Xva)[:, 1]
            else:
                model.fit(Xtr, y_train_seed)
                yva_prob = model.predict_proba(Xva)[:, 1]
            t1 = time.perf_counter()

            auc = roc_auc_score(y_valid_seed, yva_prob)
            if verbose:
                print(
                    f"[{model_name}|seed={seed}] trial={trial.number} auc={auc:.6f} "
                    f"train_time={t1-t0:.2f}s"
                )
            return auc

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, callbacks=[optuna_log_callback])

        best_params = study.best_trial.params
        print(f"[BEST] {model_name} seed={seed} best_auc(valid)={study.best_value:.6f}")
        print(f"[BEST] params={best_params}")

        final_model = model_builder_fn(best_params, seed)
        t0 = time.perf_counter()
        if use_native_cat:
            final_model.fit(
                Xtr, y_train_seed,
                eval_set=(Xva, y_valid_seed),
                cat_features=cat_cols,
                use_best_model=True,
                verbose=100
            )
            y_prob_test = final_model.predict_proba(Xte)[:, 1]
            infer_mean, infer_std = benchmark_inference(
                lambda x: final_model.predict_proba(x)[:, 1], Xte
            )
            model_size = estimate_model_size_mb(final_model)
            preprocess_obj = None
        else:
            final_model.fit(Xtr, y_train_seed)
            y_prob_test = final_model.predict_proba(Xte)[:, 1]
            infer_mean, infer_std = benchmark_inference(
                lambda x: final_model.predict_proba(preprocessor.transform(x))[:, 1],
                X_test
            )
            try:
                model_size = estimate_model_size_mb((preprocessor, final_model))
            except Exception:
                model_size = np.nan
            preprocess_obj = preprocessor
        t1 = time.perf_counter()

        m = evaluate_predictions(y_test, y_prob_test)
        m["train_seconds"] = float(t1 - t0)
        m["infer_ms_per_sample"] = float(infer_mean)
        m["infer_ms_per_sample_std"] = float(infer_std)
        m["model_size_mb"] = float(model_size)
        m["best_valid_auc"] = float(study.best_value)
        m["best_params"] = best_params
        seed_results.append(m)

        if m["auc_roc"] > best_auc:
            best_auc = m["auc_roc"]
            best_pack = (final_model, preprocess_obj, seed, best_params)

        gc.collect()

    all_results[model_name] = seed_results
    best_models[model_name] = best_pack[0]
    best_preprocessors[model_name] = best_pack[1]
    print_seed_metrics(model_name, seed_results)

In [10]:
# ===== 3.1 Logistic Regression =====
def logistic_builder(params, seed):
    return LogisticRegression(
        C=params["C"],
        penalty=params["penalty"],
        solver="saga",
        max_iter=3000,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )

def logistic_suggester(trial):
    return {
        "C": trial.suggest_float("C", 1e-3, 30.0, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
    }

run_classical_experiment(
    model_name="LogisticRegression",
    model_builder_fn=logistic_builder,
    param_suggester_fn=logistic_suggester,
    model_family="linear",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== LogisticRegression | seed=123 ====================


[I 2026-04-04 10:17:04,872] A new study created in memory with name: no-name-615a1821-4a58-4a00-9bba-45616c724616
[I 2026-04-04 10:17:09,782] Trial 0 finished with value: 0.618930711091375 and parameters: {'C': 0.0038782012380407013, 'penalty': 'l2'}. Best is trial 0 with value: 0.618930711091375.


[LogisticRegression|seed=123] trial=0 auc=0.618931 train_time=4.89s
[Optuna] Trial 000 | value=0.618931 | params={'C': 0.0038782012380407013, 'penalty': 'l2'}
[BEST] LogisticRegression seed=123 best_auc(valid)=0.618931
[BEST] params={'C': 0.0038782012380407013, 'penalty': 'l2'}

[SUMMARY] LogisticRegression
  - accuracy            : mean=0.628118, std=0.000000
  - auc_roc             : mean=0.619407, std=0.000000
  - f1                  : mean=0.094646, std=0.000000
  - train_seconds       : mean=6.868416, std=0.000000
  - infer_ms_per_sample : mean=0.007285, std=0.000000
  - model_size_mb       : mean=0.010950, std=0.000000


In [11]:
# ===== 3.2 Random Forest =====
def rf_builder(params, seed):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=seed,
    )

def rf_suggester(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
        "max_depth": trial.suggest_int("max_depth", 4, 24),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }

run_classical_experiment(
    model_name="RandomForest",
    model_builder_fn=rf_builder,
    param_suggester_fn=rf_suggester,
    model_family="tree",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== RandomForest | seed=123 ====================


[I 2026-04-04 10:17:17,715] A new study created in memory with name: no-name-09f031c3-a7f8-4097-8643-3bec134b51ca
[I 2026-04-04 10:19:26,652] Trial 0 finished with value: 0.6141414007768682 and parameters: {'n_estimators': 900, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.6141414007768682.


[RandomForest|seed=123] trial=0 auc=0.614141 train_time=128.91s
[Optuna] Trial 000 | value=0.614141 | params={'n_estimators': 900, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None}
[BEST] RandomForest seed=123 best_auc(valid)=0.614141
[BEST] params={'n_estimators': 900, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None}

[SUMMARY] RandomForest
  - accuracy            : mean=0.828600, std=0.000000
  - auc_roc             : mean=0.600682, std=0.000000
  - f1                  : mean=0.093075, std=0.000000
  - train_seconds       : mean=171.545205, std=0.000000
  - infer_ms_per_sample : mean=0.186520, std=0.000000
  - model_size_mb       : mean=22.510600, std=0.000000


In [12]:
# ===== 3.3 XGBoost =====
def xgb_builder(params, seed):
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        n_estimators=params["n_estimators"],
        random_state=seed,
        n_jobs=-1,
    )

def xgb_suggester(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1400, step=100),
    }

run_classical_experiment(
    model_name="XGBoost",
    model_builder_fn=xgb_builder,
    param_suggester_fn=xgb_suggester,
    model_family="tree",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== XGBoost | seed=123 ====================


[I 2026-04-04 10:22:19,519] A new study created in memory with name: no-name-2ce9b573-a19f-4f35-b768-edd1dd2688d2
[I 2026-04-04 10:22:28,594] Trial 0 finished with value: 0.5800990443155228 and parameters: {'learning_rate': 0.16632702948566025, 'max_depth': 4, 'min_child_weight': 0.03706745046083238, 'subsample': 0.8702765485608672, 'colsample_bytree': 0.84449302921763, 'reg_alpha': 0.0007222699104883079, 'reg_lambda': 2.396517531238659e-07, 'n_estimators': 1300}. Best is trial 0 with value: 0.5800990443155228.


[XGBoost|seed=123] trial=0 auc=0.580099 train_time=9.05s
[Optuna] Trial 000 | value=0.580099 | params={'learning_rate': 0.16632702948566025, 'max_depth': 4, 'min_child_weight': 0.03706745046083238, 'subsample': 0.8702765485608672, 'colsample_bytree': 0.84449302921763, 'reg_alpha': 0.0007222699104883079, 'reg_lambda': 2.396517531238659e-07, 'n_estimators': 1300}
[BEST] XGBoost seed=123 best_auc(valid)=0.580099
[BEST] params={'learning_rate': 0.16632702948566025, 'max_depth': 4, 'min_child_weight': 0.03706745046083238, 'subsample': 0.8702765485608672, 'colsample_bytree': 0.84449302921763, 'reg_alpha': 0.0007222699104883079, 'reg_lambda': 2.396517531238659e-07, 'n_estimators': 1300}

[SUMMARY] XGBoost
  - accuracy            : mean=0.961602, std=0.000000
  - auc_roc             : mean=0.555233, std=0.000000
  - f1                  : mean=0.005656, std=0.000000
  - train_seconds       : mean=10.094875, std=0.000000
  - infer_ms_per_sample : mean=0.016789, std=0.000000
  - model_size_mb    

In [13]:
# ===== 3.4 LightGBM =====
def lgbm_builder(params, seed):
    return LGBMClassifier(
        objective="binary",
        metric="auc",
        learning_rate=params["learning_rate"],
        n_estimators=params["n_estimators"],
        num_leaves=params["num_leaves"],
        max_depth=params["max_depth"],
        min_child_samples=params["min_child_samples"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        random_state=seed,
        n_jobs=-1,
        verbosity=1,
    )

def lgbm_suggester(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1800, step=100),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

run_classical_experiment(
    model_name="LightGBM",
    model_builder_fn=lgbm_builder,
    param_suggester_fn=lgbm_suggester,
    model_family="tree",
    use_native_cat=False,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== LightGBM | seed=123 ====================


[I 2026-04-04 10:22:40,014] A new study created in memory with name: no-name-1c270064-af71-4e51-adaf-7f5b3bb9d2a8


[LightGBM] [Info] Number of positive: 2884, number of negative: 77116
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007103 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1583
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036050 -> initscore=-3.286133
[LightGBM] [Info] Start training from score -3.286133
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[I 2026-04-04 10:22:59,705] Trial 0 finished with value: 0.6049273649599018 and parameters: {'learning_rate': 0.019568105267349197, 'n_estimators': 1400, 'num_leaves': 246, 'max_depth': 8, 'min_child_samples': 51, 'subsample': 0.6361639760595841, 'colsample_bytree': 0.6007313248564923, 'reg_alpha': 0.0012774078708259861, 'reg_lambda': 1.8729851653148345e-08}. Best is trial 0 with value: 0.6049273649599018.


[LightGBM|seed=123] trial=0 auc=0.604927 train_time=19.65s
[Optuna] Trial 000 | value=0.604927 | params={'learning_rate': 0.019568105267349197, 'n_estimators': 1400, 'num_leaves': 246, 'max_depth': 8, 'min_child_samples': 51, 'subsample': 0.6361639760595841, 'colsample_bytree': 0.6007313248564923, 'reg_alpha': 0.0012774078708259861, 'reg_lambda': 1.8729851653148345e-08}
[BEST] LightGBM seed=123 best_auc(valid)=0.604927
[BEST] params={'learning_rate': 0.019568105267349197, 'n_estimators': 1400, 'num_leaves': 246, 'max_depth': 8, 'min_child_samples': 51, 'subsample': 0.6361639760595841, 'colsample_bytree': 0.6007313248564923, 'reg_alpha': 0.0012774078708259861, 'reg_lambda': 1.8729851653148345e-08}
[LightGBM] [Info] Number of positive: 2884, number of negative: 77116
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011700 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`

In [14]:
# ===== 3.5 CatBoost =====
def cat_builder(params, seed):
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],
        iterations=params["iterations"],
        random_seed=seed,
        verbose=100,
    )

def cat_suggester(trial):
    return {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        "iterations": trial.suggest_int("iterations", 300, 2000, step=100),
    }

run_classical_experiment(
    model_name="CatBoost",
    model_builder_fn=cat_builder,
    param_suggester_fn=cat_suggester,
    model_family="tree",
    use_native_cat=True,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    verbose=True,
    )


==================== CatBoost | seed=123 ====================


[I 2026-04-04 10:23:30,741] A new study created in memory with name: no-name-9fa5461b-da3f-4e47-8df9-bab415139da4


0:	test: 0.5258634	best: 0.5258634 (0)	total: 261ms	remaining: 4m 46s
100:	test: 0.6211372	best: 0.6249494 (34)	total: 35.6s	remaining: 5m 52s
200:	test: 0.6006837	best: 0.6249494 (34)	total: 1m 12s	remaining: 5m 23s
300:	test: 0.5958534	best: 0.6249494 (34)	total: 1m 47s	remaining: 4m 45s
400:	test: 0.5921224	best: 0.6249494 (34)	total: 2m 27s	remaining: 4m 16s
500:	test: 0.5811825	best: 0.6249494 (34)	total: 3m 6s	remaining: 3m 43s
600:	test: 0.5725331	best: 0.6249494 (34)	total: 3m 42s	remaining: 3m 4s
700:	test: 0.5722444	best: 0.6249494 (34)	total: 4m 21s	remaining: 2m 29s
800:	test: 0.5698819	best: 0.6249494 (34)	total: 4m 58s	remaining: 1m 51s
900:	test: 0.5696010	best: 0.6249494 (34)	total: 5m 31s	remaining: 1m 13s
1000:	test: 0.5710238	best: 0.6249494 (34)	total: 6m 5s	remaining: 36.1s
1099:	test: 0.5719414	best: 0.6249494 (34)	total: 6m 32s	remaining: 0us

bestTest = 0.6249493619
bestIteration = 34

Shrink model to first 35 iterations.


[I 2026-04-04 10:30:04,788] Trial 0 finished with value: 0.6249493619444205 and parameters: {'depth': 10, 'learning_rate': 0.14902068890939277, 'l2_leaf_reg': 7.9728472001785855, 'iterations': 1100}. Best is trial 0 with value: 0.6249493619444205.


[CatBoost|seed=123] trial=0 auc=0.624949 train_time=394.02s
[Optuna] Trial 000 | value=0.624949 | params={'depth': 10, 'learning_rate': 0.14902068890939277, 'l2_leaf_reg': 7.9728472001785855, 'iterations': 1100}
[BEST] CatBoost seed=123 best_auc(valid)=0.624949
[BEST] params={'depth': 10, 'learning_rate': 0.14902068890939277, 'l2_leaf_reg': 7.9728472001785855, 'iterations': 1100}
0:	test: 0.5258634	best: 0.5258634 (0)	total: 69.6ms	remaining: 1m 16s
100:	test: 0.6211372	best: 0.6249494 (34)	total: 22.6s	remaining: 3m 43s
200:	test: 0.6006837	best: 0.6249494 (34)	total: 48.7s	remaining: 3m 37s
300:	test: 0.5958534	best: 0.6249494 (34)	total: 1m 23s	remaining: 3m 42s
400:	test: 0.5921224	best: 0.6249494 (34)	total: 2m	remaining: 3m 29s
500:	test: 0.5811825	best: 0.6249494 (34)	total: 2m 32s	remaining: 3m 2s
600:	test: 0.5725331	best: 0.6249494 (34)	total: 3m 4s	remaining: 2m 33s
700:	test: 0.5722444	best: 0.6249494 (34)	total: 3m 41s	remaining: 2m 5s
800:	test: 0.5698819	best: 0.6249494 

In [15]:
# ===== 3.6 深度模型辅助：TabNet / FT-Transformer 预处理与训练工具 =====
# FT-Transformer implementation package
try:
    from tab_transformer_pytorch import FTTransformer
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tab-transformer-pytorch"])
    from tab_transformer_pytorch import FTTransformer

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cpu" if FT_FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Torch device: {DEVICE}")

def prepare_deep_inputs(Xtr_df, Xva_df, Xte_df):
    """返回可用于 TabNet/FT-Transformer 的编码后数据。"""
    numbin_cols = num_cols + bin_cols

    # 数值：中位数填充
    num_imputer = SimpleImputer(strategy="median")
    Xtr_num = num_imputer.fit_transform(Xtr_df[numbin_cols])
    Xva_num = num_imputer.transform(Xva_df[numbin_cols])
    Xte_num = num_imputer.transform(Xte_df[numbin_cols])

    # 类别：字符串化 + 缺失占位 + Ordinal 编码（unknown -> -1, 再整体 +1）
    if len(cat_cols) > 0:
        tr_cat = Xtr_df[cat_cols].astype("string").fillna("MISSING")
        va_cat = Xva_df[cat_cols].astype("string").fillna("MISSING")
        te_cat = Xte_df[cat_cols].astype("string").fillna("MISSING")

        cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        Xtr_cat = cat_encoder.fit_transform(tr_cat).astype(np.int64) + 1
        Xva_cat = cat_encoder.transform(va_cat).astype(np.int64) + 1
        Xte_cat = cat_encoder.transform(te_cat).astype(np.int64) + 1

        cat_dims = [int(Xtr_cat[:, i].max() + 1) for i in range(Xtr_cat.shape[1])]
    else:
        Xtr_cat = np.zeros((len(Xtr_df), 0), dtype=np.int64)
        Xva_cat = np.zeros((len(Xva_df), 0), dtype=np.int64)
        Xte_cat = np.zeros((len(Xte_df), 0), dtype=np.int64)
        cat_dims = []
        cat_encoder = None

    bundle = {
        "Xtr_num": Xtr_num.astype(np.float32),
        "Xva_num": Xva_num.astype(np.float32),
        "Xte_num": Xte_num.astype(np.float32),
        "Xtr_cat": Xtr_cat,
        "Xva_cat": Xva_cat,
        "Xte_cat": Xte_cat,
        "cat_dims": cat_dims,
        "num_imputer": num_imputer,
        "cat_encoder": cat_encoder,
        "n_num": len(numbin_cols),
        "n_cat": len(cat_cols),
    }
    return bundle

def fit_fttransformer_once(bundle, ytr, yva, params, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = FTTransformer(
        categories=tuple(bundle["cat_dims"]),
        num_continuous=bundle["n_num"],
        dim=params["dim"],
        dim_out=1,
        depth=params["depth"],
        heads=params["heads"],
        attn_dropout=params["attn_dropout"],
        ff_dropout=params["ff_dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()

    xtr_cat = torch.tensor(bundle["Xtr_cat"], dtype=torch.long)
    xtr_num = torch.tensor(bundle["Xtr_num"], dtype=torch.float32)
    ytr_t = torch.tensor(np.asarray(ytr).reshape(-1, 1), dtype=torch.float32)

    train_ds = TensorDataset(xtr_cat, xtr_num, ytr_t)
    train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)

    xva_cat = torch.tensor(bundle["Xva_cat"], dtype=torch.long, device=DEVICE)
    xva_num = torch.tensor(bundle["Xva_num"], dtype=torch.float32, device=DEVICE)
    yva_np = np.asarray(yva).astype(int)

    best_auc = -np.inf
    best_state = None
    no_improve = 0

    for epoch in range(1, params["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for bc, bn, by in train_loader:
            bc = bc.to(DEVICE)
            bn = bn.to(DEVICE)
            by = by.to(DEVICE)

            optimizer.zero_grad()
            logits = model(bc, bn)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(by)

        model.eval()
        with torch.no_grad():
            logits_va = model(xva_cat, xva_num).squeeze(1)
            prob_va = torch.sigmoid(logits_va).detach().cpu().numpy()
        auc_va = roc_auc_score(yva_np, prob_va)
        avg_loss = running_loss / len(train_ds)

        print(f"[FTTransformer|seed={seed}] epoch={epoch:03d} loss={avg_loss:.6f} valid_auc={auc_va:.6f}")

        if auc_va > best_auc:
            best_auc = auc_va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= params["patience"]:
            print(f"[FTTransformer|seed={seed}] Early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, float(best_auc)

def ft_predict_proba(model, X_cat, X_num, batch_size=2048):
    model.eval()
    probs = []
    n = len(X_num)
    with torch.no_grad():
        for i in range(0, n, batch_size):
            bc = torch.tensor(X_cat[i:i+batch_size], dtype=torch.long, device=DEVICE)
            bn = torch.tensor(X_num[i:i+batch_size], dtype=torch.float32, device=DEVICE)
            logits = model(bc, bn).squeeze(1)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)

[INFO] Torch device: cpu


In [16]:
# ===== 3.6 TabNet =====
def run_tabnet_experiment(n_trials=N_TRIALS, seeds=SEEDS):
    model_name = "TabNet"
    seed_results = []
    best_auc = -np.inf
    best_pack = None

    for seed in seeds:
        print(f"\n{'='*20} {model_name} | seed={seed} {'='*20}")
        np.random.seed(seed)
        random.seed(seed)
        torch.manual_seed(seed)

        X_train_seed = X_train
        y_train_seed = y_train
        X_valid_seed = X_valid
        y_valid_seed = y_valid

        if MAX_TRAIN_ROWS is not None and len(X_train_seed) > MAX_TRAIN_ROWS:
            idx = X_train_seed.sample(n=MAX_TRAIN_ROWS, random_state=seed).index
            X_train_seed = X_train_seed.loc[idx]
            y_train_seed = y_train_seed.loc[idx]

        if MAX_VALID_ROWS is not None and len(X_valid_seed) > MAX_VALID_ROWS:
            idx = X_valid_seed.sample(n=MAX_VALID_ROWS, random_state=seed).index
            X_valid_seed = X_valid_seed.loc[idx]
            y_valid_seed = y_valid_seed.loc[idx]

        bundle = prepare_deep_inputs(X_train_seed, X_valid_seed, X_test)

        Xtr_tab = np.hstack([bundle["Xtr_num"], bundle["Xtr_cat"]]).astype(np.float32)
        Xva_tab = np.hstack([bundle["Xva_num"], bundle["Xva_cat"]]).astype(np.float32)
        Xte_tab = np.hstack([bundle["Xte_num"], bundle["Xte_cat"]]).astype(np.float32)

        cat_idxs = list(range(bundle["n_num"], bundle["n_num"] + bundle["n_cat"]))
        cat_dims = bundle["cat_dims"]

        ytr_np = y_train_seed.values
        yva_np = y_valid_seed.values
        yte_np = y_test.values

        def objective(trial):
            params = {
                "n_d": trial.suggest_int("n_d", 8, 64, step=8),
                "n_a": trial.suggest_int("n_a", 8, 64, step=8),
                "n_steps": trial.suggest_int("n_steps", 3, 8),
                "gamma": trial.suggest_float("gamma", 1.0, 2.0),
                "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-7, 1e-2, log=True),
                "lr": trial.suggest_float("lr", 1e-4, 5e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [1024, 2048, 4096]),
                "virtual_batch_size": trial.suggest_categorical("virtual_batch_size", [128, 256, 512]),
            }

            model = TabNetClassifier(
                n_d=params["n_d"],
                n_a=params["n_a"],
                n_steps=params["n_steps"],
                gamma=params["gamma"],
                lambda_sparse=params["lambda_sparse"],
                cat_idxs=cat_idxs,
                cat_dims=cat_dims,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=params["lr"]),
                seed=seed,
                verbose=10,
            )

            t0 = time.perf_counter()
            model.fit(
                X_train=Xtr_tab, y_train=ytr_np,
                eval_set=[(Xva_tab, yva_np)],
                eval_name=["valid"],
                eval_metric=["auc"],
                max_epochs=TABNET_MAX_EPOCHS,
                patience=min(12, max(5, TABNET_MAX_EPOCHS // 4)),
                batch_size=params["batch_size"],
                virtual_batch_size=params["virtual_batch_size"],
                drop_last=False,
            )
            t1 = time.perf_counter()

            yva_prob = model.predict_proba(Xva_tab)[:, 1]
            auc = roc_auc_score(yva_np, yva_prob)
            print(f"[TabNet|seed={seed}] trial={trial.number} auc={auc:.6f} train_time={t1-t0:.2f}s")
            return auc

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, callbacks=[optuna_log_callback])
        best_params = study.best_trial.params
        print(f"[BEST] TabNet seed={seed} best_valid_auc={study.best_value:.6f}")
        print(f"[BEST] params={best_params}")

        final_model = TabNetClassifier(
            n_d=best_params["n_d"],
            n_a=best_params["n_a"],
            n_steps=best_params["n_steps"],
            gamma=best_params["gamma"],
            lambda_sparse=best_params["lambda_sparse"],
            cat_idxs=cat_idxs,
            cat_dims=cat_dims,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=best_params["lr"]),
            seed=seed,
            verbose=10,
        )

        t0 = time.perf_counter()
        final_model.fit(
            X_train=Xtr_tab, y_train=ytr_np,
            eval_set=[(Xva_tab, yva_np)],
            eval_name=["valid"],
            eval_metric=["auc"],
            max_epochs=TABNET_MAX_EPOCHS,
            patience=min(12, max(5, TABNET_MAX_EPOCHS // 4)),
            batch_size=best_params["batch_size"],
            virtual_batch_size=best_params["virtual_batch_size"],
            drop_last=False,
        )
        y_prob_test = final_model.predict_proba(Xte_tab)[:, 1]
        t1 = time.perf_counter()

        m = evaluate_predictions(yte_np, y_prob_test)
        m["train_seconds"] = float(t1 - t0)

        # 推理测速
        lat_list = []
        for _ in range(INFER_REPEAT):
            start = time.perf_counter()
            _ = final_model.predict_proba(Xte_tab)[:, 1]
            end = time.perf_counter()
            lat_list.append((end - start) / len(Xte_tab) * 1000)
        m["infer_ms_per_sample"] = float(np.mean(lat_list))
        m["infer_ms_per_sample_std"] = float(np.std(lat_list))

        # 模型体积估计（TabNet.save_model 需要目录前缀，不是现有文件路径）
        import tempfile
        with tempfile.TemporaryDirectory() as tmpdir:
            save_prefix = os.path.join(tmpdir, "tabnet_model")
            saved_path = final_model.save_model(save_prefix)
            zip_path = saved_path if str(saved_path).endswith(".zip") else f"{saved_path}.zip"
            if not os.path.exists(zip_path):
                zip_path = f"{save_prefix}.zip"
            size_mb = os.path.getsize(zip_path) / (1024 * 1024)

        m["model_size_mb"] = float(size_mb)
        m["best_valid_auc"] = float(study.best_value)
        m["best_params"] = best_params
        seed_results.append(m)

        if m["auc_roc"] > best_auc:
            best_auc = m["auc_roc"]
            best_pack = (final_model, bundle, seed, best_params)

        gc.collect()

    all_results[model_name] = seed_results
    best_models[model_name] = best_pack[0]
    best_preprocessors[model_name] = best_pack[1]  # deep bundle
    print_seed_metrics(model_name, seed_results)

run_tabnet_experiment()


==================== TabNet | seed=123 ====================


[I 2026-04-04 10:36:07,784] A new study created in memory with name: no-name-05d527bb-2859-46b1-9053-37496df8182b


epoch 0  | loss: 1.17393 | valid_auc: 0.51056 |  0:00:06s
epoch 10 | loss: 0.27037 | valid_auc: 0.50412 |  0:01:33s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_valid_auc = 0.51056


[I 2026-04-04 10:37:50,784] Trial 0 finished with value: 0.5105646136688862 and parameters: {'n_d': 8, 'n_a': 48, 'n_steps': 3, 'gamma': 1.0407989925599819, 'lambda_sparse': 3.261737152008134e-07, 'lr': 0.0002035279668067561, 'batch_size': 2048, 'virtual_batch_size': 256}. Best is trial 0 with value: 0.5105646136688862.


[TabNet|seed=123] trial=0 auc=0.510565 train_time=101.73s
[Optuna] Trial 000 | value=0.510565 | params={'n_d': 8, 'n_a': 48, 'n_steps': 3, 'gamma': 1.0407989925599819, 'lambda_sparse': 3.261737152008134e-07, 'lr': 0.0002035279668067561, 'batch_size': 2048, 'virtual_batch_size': 256}
[BEST] TabNet seed=123 best_valid_auc=0.510565
[BEST] params={'n_d': 8, 'n_a': 48, 'n_steps': 3, 'gamma': 1.0407989925599819, 'lambda_sparse': 3.261737152008134e-07, 'lr': 0.0002035279668067561, 'batch_size': 2048, 'virtual_batch_size': 256}
epoch 0  | loss: 1.17393 | valid_auc: 0.51056 |  0:00:08s
epoch 10 | loss: 0.27037 | valid_auc: 0.50412 |  0:01:20s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_valid_auc = 0.51056
Successfully saved model at C:\Users\27335\AppData\Local\Temp\tmputetxlaj\tabnet_model.zip

[SUMMARY] TabNet
  - accuracy            : mean=0.245987, std=0.000000
  - auc_roc             : mean=0.501979, std=0.000000
  - f1                  : mean=0.069478, std=0.000000
 

In [17]:
# ===== 3.7 FT-Transformer =====
def run_fttransformer_experiment(n_trials=N_TRIALS, seeds=SEEDS):
    model_name = "FTTransformer"
    seed_results = []
    best_auc = -np.inf
    best_pack = None

    for seed in seeds:
        print(f"\n{'='*20} {model_name} | seed={seed} {'='*20}")
        np.random.seed(seed)
        random.seed(seed)
        torch.manual_seed(seed)

        X_train_seed = X_train
        y_train_seed = y_train
        X_valid_seed = X_valid
        y_valid_seed = y_valid

        # FT 单独更激进降采样，避免内存/显存压力导致 kernel crash
        train_cap = MAX_TRAIN_ROWS_FT if MAX_TRAIN_ROWS_FT is not None else MAX_TRAIN_ROWS
        valid_cap = MAX_VALID_ROWS_FT if MAX_VALID_ROWS_FT is not None else MAX_VALID_ROWS

        if train_cap is not None and len(X_train_seed) > train_cap:
            idx = X_train_seed.sample(n=train_cap, random_state=seed).index
            X_train_seed = X_train_seed.loc[idx]
            y_train_seed = y_train_seed.loc[idx]

        if valid_cap is not None and len(X_valid_seed) > valid_cap:
            idx = X_valid_seed.sample(n=valid_cap, random_state=seed).index
            X_valid_seed = X_valid_seed.loc[idx]
            y_valid_seed = y_valid_seed.loc[idx]

        bundle = prepare_deep_inputs(X_train_seed, X_valid_seed, X_test)

        ytr_np = y_train_seed.values.astype(int)
        yva_np = y_valid_seed.values.astype(int)
        yte_np = y_test.values.astype(int)

        def objective(trial):
            epoch_high = max(1, int(FT_MAX_EPOCHS))
            epoch_low = 1 if epoch_high < 8 else 8
            patience_high = min(6, epoch_high)
            patience_low = 1 if patience_high < 3 else 3

            params = {
                "dim": trial.suggest_categorical("dim", [16, 24, 32]),
                "depth": trial.suggest_int("depth", 2, 4),
                "heads": trial.suggest_categorical("heads", [4]),
                "attn_dropout": trial.suggest_float("attn_dropout", 0.0, 0.3),
                "ff_dropout": trial.suggest_float("ff_dropout", 0.0, 0.3),
                "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
                "epochs": trial.suggest_int("epochs", epoch_low, epoch_high),
                "patience": trial.suggest_int("patience", patience_low, patience_high),
            }
            t0 = time.perf_counter()
            model, best_auc_valid = fit_fttransformer_once(bundle, ytr_np, yva_np, params, seed)
            t1 = time.perf_counter()
            print(
                f"[FTTransformer|seed={seed}] trial={trial.number} valid_auc={best_auc_valid:.6f} "
                f"train_time={t1-t0:.2f}s"
            )
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return best_auc_valid

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, callbacks=[optuna_log_callback])
        best_params = study.best_trial.params
        print(f"[BEST] FTTransformer seed={seed} best_valid_auc={study.best_value:.6f}")
        print(f"[BEST] params={best_params}")

        t0 = time.perf_counter()
        final_model, _ = fit_fttransformer_once(bundle, ytr_np, yva_np, best_params, seed)
        y_prob_test = ft_predict_proba(final_model, bundle["Xte_cat"], bundle["Xte_num"])
        t1 = time.perf_counter()

        m = evaluate_predictions(yte_np, y_prob_test)
        m["train_seconds"] = float(t1 - t0)

        lat_list = []
        for _ in range(INFER_REPEAT):
            start = time.perf_counter()
            _ = ft_predict_proba(final_model, bundle["Xte_cat"], bundle["Xte_num"])
            end = time.perf_counter()
            lat_list.append((end - start) / len(bundle["Xte_num"]) * 1000)
        m["infer_ms_per_sample"] = float(np.mean(lat_list))
        m["infer_ms_per_sample_std"] = float(np.std(lat_list))

        # 模型体积（state_dict）
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as tmp:
            path = tmp.name
        torch.save(final_model.state_dict(), path)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        try:
            os.remove(path)
        except Exception:
            pass
        m["model_size_mb"] = float(size_mb)
        m["best_valid_auc"] = float(study.best_value)
        m["best_params"] = best_params
        seed_results.append(m)

        if m["auc_roc"] > best_auc:
            best_auc = m["auc_roc"]
            best_pack = (final_model, bundle, seed, best_params)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    all_results[model_name] = seed_results
    best_models[model_name] = best_pack[0]
    best_preprocessors[model_name] = best_pack[1]  # deep bundle
    print_seed_metrics(model_name, seed_results)

run_fttransformer_experiment()


==================== FTTransformer | seed=123 ====================


[I 2026-04-04 10:39:29,854] A new study created in memory with name: no-name-937b8239-2ff8-47b6-8225-198bc8d07dae


[FTTransformer|seed=123] epoch=001 loss=0.178889 valid_auc=0.499082
[FTTransformer|seed=123] epoch=002 loss=0.157253 valid_auc=0.523850
[FTTransformer|seed=123] epoch=003 loss=0.157061 valid_auc=0.532008
[FTTransformer|seed=123] epoch=004 loss=0.156959 valid_auc=0.539023
[FTTransformer|seed=123] epoch=005 loss=0.156872 valid_auc=0.553794
[FTTransformer|seed=123] epoch=006 loss=0.156423 valid_auc=0.531124
[FTTransformer|seed=123] epoch=007 loss=0.155525 valid_auc=0.571803
[FTTransformer|seed=123] epoch=008 loss=0.154899 valid_auc=0.555298
[FTTransformer|seed=123] trial=0 valid_auc=0.571803 train_time=4799.17s


[I 2026-04-04 11:59:29,817] Trial 0 finished with value: 0.5718026364643198 and parameters: {'dim': 24, 'depth': 4, 'heads': 4, 'attn_dropout': 0.1566744927358368, 'ff_dropout': 0.13726733674444175, 'lr': 0.0020310335578301777, 'weight_decay': 4.223950634808983e-06, 'batch_size': 64, 'epochs': 8, 'patience': 6}. Best is trial 0 with value: 0.5718026364643198.


[Optuna] Trial 000 | value=0.571803 | params={'dim': 24, 'depth': 4, 'heads': 4, 'attn_dropout': 0.1566744927358368, 'ff_dropout': 0.13726733674444175, 'lr': 0.0020310335578301777, 'weight_decay': 4.223950634808983e-06, 'batch_size': 64, 'epochs': 8, 'patience': 6}
[BEST] FTTransformer seed=123 best_valid_auc=0.571803
[BEST] params={'dim': 24, 'depth': 4, 'heads': 4, 'attn_dropout': 0.1566744927358368, 'ff_dropout': 0.13726733674444175, 'lr': 0.0020310335578301777, 'weight_decay': 4.223950634808983e-06, 'batch_size': 64, 'epochs': 8, 'patience': 6}
[FTTransformer|seed=123] epoch=001 loss=0.178889 valid_auc=0.499082
[FTTransformer|seed=123] epoch=002 loss=0.157253 valid_auc=0.523850
[FTTransformer|seed=123] epoch=003 loss=0.157061 valid_auc=0.532008
[FTTransformer|seed=123] epoch=004 loss=0.156959 valid_auc=0.539023
[FTTransformer|seed=123] epoch=005 loss=0.156872 valid_auc=0.553794
[FTTransformer|seed=123] epoch=006 loss=0.156423 valid_auc=0.531124
[FTTransformer|seed=123] epoch=007 lo

## 4) 模型预测与综合分析

本阶段产出：
- 各模型在 test 集上的均值±标准差结果总表
- 推理成本与模型体积对比
- 结果稳定性（跨 seed 波动）
- 特征重要性一致性对比
- 面向实践的结论建议

In [18]:
# ===== 4.1 汇总结果表 =====
rows = []
for model_name, seed_res in all_results.items():
    if len(seed_res) == 0:
        continue
    row = {"model": model_name}
    for m in ["accuracy", "auc_roc", "f1", "train_seconds", "infer_ms_per_sample", "model_size_mb", "best_valid_auc"]:
        vals = [r[m] for r in seed_res]
        row[f"{m}_mean"] = float(np.mean(vals))
        row[f"{m}_std"] = float(np.std(vals))
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values("auc_roc_mean", ascending=False).reset_index(drop=True)
display(summary_df)

summary_path = RESULT_DIR / "model_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"[INFO] Saved summary to {summary_path}")

,model,accuracy_mean,accuracy_std,auc_roc_mean,auc_roc_std,f1_mean,f1_std,train_seconds_mean,train_seconds_std,infer_ms_per_sample_mean,infer_ms_per_sample_std,model_size_mb_mean,model_size_mb_std,best_valid_auc_mean,best_valid_auc_std
0,LogisticRegression,0.63,0.0,0.62,0.0,9.46e-02,0.0,6.87,0.0,7.29e-03,0.0,0.01,0.0,0.62,0.0
1,CatBoost,0.96,0.0,0.62,0.0,0.00e+00,0.0,359.24,0.0,1.65e-02,0.0,0.78,0.0,0.62,0.0
2,RandomForest,0.83,0.0,0.60,0.0,9.31e-02,0.0,171.55,0.0,1.87e-01,0.0,22.51,0.0,0.61,0.0
3,LightGBM,0.96,0.0,0.58,0.0,0.00e+00,0.0,28.98,0.0,5.95e-02,0.0,12.59,0.0,0.60,0.0
4,FTTransformer,0.96,0.0,0.58,0.0,0.00e+00,0.0,5684.17,0.0,1.53e+01,0.0,0.27,0.0,0.57,0.0
5,XGBoost,0.96,0.0,0.56,0.0,5.66e-03,0.0,10.09,0.0,1.68e-02,0.0,2.10,0.0,0.58,0.0
6,TabNet,0.25,0.0,0.50,0.0,6.95e-02,0.0,91.38,0.0,4.67e-02,0.0,0.29,0.0,0.51,0.0


[INFO] Saved summary to artifacts\model_summary.csv


In [19]:
# ===== 4.2 输出各模型测试集预测（概率） =====
test_pred_df = pd.DataFrame(index=X_test.index)
test_pred_df[TARGET_COL] = y_test.values

for model_name, model in best_models.items():
    print(f"[PREDICT] {model_name}")
    if model_name in ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]:
        prep = best_preprocessors[model_name]
        proba = model.predict_proba(prep.transform(X_test))[:, 1]
    elif model_name == "CatBoost":
        Xt = X_test.copy()
        for c in cat_cols:
            Xt[c] = Xt[c].astype("string").fillna("MISSING")
        proba = model.predict_proba(Xt)[:, 1]
    elif model_name in ["TabNet"]:
        bundle = best_preprocessors[model_name]
        Xte_tab = np.hstack([bundle["Xte_num"], bundle["Xte_cat"]]).astype(np.float32)
        proba = model.predict_proba(Xte_tab)[:, 1]
    elif model_name in ["FTTransformer"]:
        bundle = best_preprocessors[model_name]
        proba = ft_predict_proba(model, bundle["Xte_cat"], bundle["Xte_num"])
    else:
        continue
    test_pred_df[f"proba_{model_name}"] = proba

pred_path = RESULT_DIR / "test_predictions_all_models.csv"
test_pred_df.to_csv(pred_path, index=True)
print(f"[INFO] Saved test predictions to {pred_path}")
display(test_pred_df.head())

[PREDICT] LogisticRegression
[PREDICT] RandomForest
[PREDICT] XGBoost
[PREDICT] LightGBM
[PREDICT] CatBoost
[PREDICT] TabNet
[PREDICT] FTTransformer
[INFO] Saved test predictions to artifacts\test_predictions_all_models.csv


,target,proba_LogisticRegression,proba_RandomForest,proba_XGBoost,proba_LightGBM,proba_CatBoost,proba_TabNet,proba_FTTransformer
235465,0,0.38,0.42,6.46e-04,1.31e-02,0.03,0.58,0.03
292820,0,0.48,0.53,1.91e-01,8.70e-02,0.04,0.67,0.03
434381,0,0.44,0.41,4.80e-03,9.47e-03,0.03,0.51,0.03
472579,0,0.46,0.42,2.45e-02,2.84e-02,0.03,0.99,0.03
238219,0,0.57,0.51,5.30e-03,1.32e-02,0.07,0.57,0.03


In [20]:
# ===== 4.3 预处理成本与鲁棒性分析 =====
preprocess_notes = pd.DataFrame([
    {"model": "LogisticRegression", "missing_handling": "median/mode imputation", "categorical": "one-hot", "effort_level": "high"},
    {"model": "RandomForest", "missing_handling": "median/mode imputation", "categorical": "ordinal encoding", "effort_level": "medium"},
    {"model": "XGBoost", "missing_handling": "native missing + imputation in this pipeline", "categorical": "ordinal encoding", "effort_level": "medium"},
    {"model": "LightGBM", "missing_handling": "native missing + imputation in this pipeline", "categorical": "ordinal encoding", "effort_level": "medium"},
    {"model": "CatBoost", "missing_handling": "native support", "categorical": "native categorical", "effort_level": "low"},
    {"model": "TabNet", "missing_handling": "numeric imputation + missing category token", "categorical": "learned embeddings", "effort_level": "high"},
    {"model": "FTTransformer", "missing_handling": "numeric imputation + missing category token", "categorical": "learned embeddings", "effort_level": "high"},
])
display(preprocess_notes)

if not summary_df.empty:
    stability = summary_df[["model", "auc_roc_std", "f1_std", "accuracy_std"]].sort_values("auc_roc_std")
    print("[INFO] Lower std => more stable across seeds")
    display(stability)

,model,missing_handling,categorical,effort_level
0,LogisticRegression,median/mode imputation,one-hot,high
1,RandomForest,median/mode imputation,ordinal encoding,medium
2,XGBoost,native missing + imputation in this pipeline,ordinal encoding,medium
3,LightGBM,native missing + imputation in this pipeline,ordinal encoding,medium
4,CatBoost,native support,native categorical,low
5,TabNet,numeric imputation + missing category token,learned embeddings,high
6,FTTransformer,numeric imputation + missing category token,learned embeddings,high


[INFO] Lower std => more stable across seeds


,model,auc_roc_std,f1_std,accuracy_std
0,LogisticRegression,0.0,0.0,0.0
1,CatBoost,0.0,0.0,0.0
2,RandomForest,0.0,0.0,0.0
3,LightGBM,0.0,0.0,0.0
4,FTTransformer,0.0,0.0,0.0
5,XGBoost,0.0,0.0,0.0
6,TabNet,0.0,0.0,0.0


In [21]:
# ===== 4.4 特征重要性对比（统一置换重要性，按验证集子样本） =====
def predict_proba_on_raw(model_name, Xraw):
    model = best_models[model_name]
    if model_name in ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]:
        prep = best_preprocessors[model_name]
        return model.predict_proba(prep.transform(Xraw))[:, 1]
    if model_name == "CatBoost":
        Xt = Xraw.copy()
        for c in cat_cols:
            Xt[c] = Xt[c].astype("string").fillna("MISSING")
        return model.predict_proba(Xt)[:, 1]
    if model_name == "TabNet":
        bundle = best_preprocessors[model_name]
        b = prepare_deep_inputs(X_train, Xraw, Xraw)
        X_tab = np.hstack([b["Xva_num"], b["Xva_cat"]]).astype(np.float32)
        return model.predict_proba(X_tab)[:, 1]
    if model_name == "FTTransformer":
        b = prepare_deep_inputs(X_train, Xraw, Xraw)
        return ft_predict_proba(model, b["Xva_cat"], b["Xva_num"])
    raise ValueError(model_name)

def permutation_importance_auc(model_name, Xraw, ytrue, max_features=None, seed=42):
    rng = np.random.default_rng(seed)
    base_pred = predict_proba_on_raw(model_name, Xraw)
    base_auc = roc_auc_score(ytrue, base_pred)
    cols = list(Xraw.columns)
    if max_features is not None and len(cols) > max_features:
        cols = cols[:max_features]
    imps = []
    for c in cols:
        Xp = Xraw.copy()
        Xp[c] = rng.permutation(Xp[c].values)
        auc_p = roc_auc_score(ytrue, predict_proba_on_raw(model_name, Xp))
        imps.append((c, base_auc - auc_p))
    imp_df = pd.DataFrame(imps, columns=["feature", "importance_drop_auc"]).sort_values(
        "importance_drop_auc", ascending=False
    )
    return base_auc, imp_df

X_val_for_imp = X_valid.sample(min(len(X_valid), 6000), random_state=SEEDS[0])
y_val_for_imp = y_valid.loc[X_val_for_imp.index]

importance_tables = {}
for model_name in all_results.keys():
    print(f"\n[IMP] {model_name}")
    base_auc, imp_df = permutation_importance_auc(model_name, X_val_for_imp, y_val_for_imp, max_features=None, seed=SEEDS[0])
    importance_tables[model_name] = imp_df
    print(f"[IMP] base_auc={base_auc:.6f}")
    display(imp_df.head(15))


[IMP] LogisticRegression
[IMP] base_auc=0.628466


,feature,importance_drop_auc
5,ps_ind_05_cat,2.44e-02
35,ps_car_13,8.60e-03
19,ps_reg_01,8.21e-03
6,ps_ind_06_bin,7.90e-03
51,ps_calc_14,6.98e-03
34,ps_car_12,5.56e-03
21,ps_reg_03,4.66e-03
32,ps_car_11_cat,4.24e-03
17,ps_ind_17_bin,4.11e-03
8,ps_ind_08_bin,4.07e-03



[IMP] RandomForest
[IMP] base_auc=0.622032


,feature,importance_drop_auc
35,ps_car_13,4.03e-02
21,ps_reg_03,1.85e-02
6,ps_ind_06_bin,1.22e-02
20,ps_reg_02,1.14e-02
19,ps_reg_01,9.38e-03
32,ps_car_11_cat,5.90e-03
15,ps_ind_15,5.54e-03
51,ps_calc_14,4.43e-03
43,ps_calc_06,4.19e-03
47,ps_calc_10,4.05e-03



[IMP] XGBoost
[IMP] base_auc=0.574647


...

,feature,importance_drop_auc
5,ps_ind_05_cat,2.03e-02
21,ps_reg_03,1.27e-02
17,ps_ind_17_bin,9.49e-03
20,ps_reg_02,7.80e-03
34,ps_car_12,6.99e-03
6,ps_ind_06_bin,6.57e-03
49,ps_calc_12,6.42e-03
44,ps_calc_07,5.13e-03
48,ps_calc_11,5.02e-03
41,ps_calc_04,4.21e-03



[IMP] TabNet
[IMP] base_auc=0.503731


,feature,importance_drop_auc
45,ps_calc_08,2.68e-02
3,ps_ind_03,2.22e-02
47,ps_calc_10,1.66e-02
15,ps_ind_15,1.53e-02
5,ps_ind_05_cat,1.35e-02
41,ps_calc_04,1.20e-02
50,ps_calc_13,1.19e-02
46,ps_calc_09,1.16e-02
33,ps_car_11,1.03e-02
9,ps_ind_09_bin,9.71e-03



[IMP] FTTransformer
[IMP] base_auc=0.571803


,feature,importance_drop_auc
5,ps_ind_05_cat,3.21e-02
24,ps_car_03_cat,3.04e-02
25,ps_car_04_cat,1.32e-02
6,ps_ind_06_bin,9.40e-03
27,ps_car_06_cat,6.64e-04
9,ps_ind_09_bin,2.90e-04
39,ps_calc_02,2.43e-04
46,ps_calc_09,2.16e-04
17,ps_ind_17_bin,1.97e-04
41,ps_calc_04,1.88e-04


In [22]:
# ===== 4.5 自动生成结论草稿（可直接用于报告） =====
if summary_df.empty:
    print("[WARN] summary_df is empty. Run training cells first.")
else:
    best_row = summary_df.iloc[0]
    fastest_row = summary_df.sort_values("infer_ms_per_sample_mean", ascending=True).iloc[0]
    smallest_row = summary_df.sort_values("model_size_mb_mean", ascending=True).iloc[0]
    most_stable_row = summary_df.sort_values("auc_roc_std", ascending=True).iloc[0]

    print("===== Practical Verdict Draft =====")
    print(f"1) Overall best predictive AUC: {best_row['model']} | AUC={best_row['auc_roc_mean']:.5f}±{best_row['auc_roc_std']:.5f}")
    print(f"2) Fastest inference: {fastest_row['model']} | {fastest_row['infer_ms_per_sample_mean']:.6f} ms/sample")
    print(f"3) Smallest model size: {smallest_row['model']} | {smallest_row['model_size_mb_mean']:.3f} MB")
    print(f"4) Most stable across seeds: {most_stable_row['model']} | AUC std={most_stable_row['auc_roc_std']:.5f}")
    print("5) Recommendation hint:")
    print("   - 若追求极致性能且可接受较高训练/预处理复杂度：优先考虑最优模型（可能是深度模型或 GBDT）。")
    print("   - 若强调部署效率、稳定性与维护成本：优先考虑 CatBoost / LightGBM / Logistic 等工程成熟方案。")
    print("   - 深度表格模型在复杂类别交互、充足样本、可投入调参资源时更可能体现优势。")

===== Practical Verdict Draft =====
1) Overall best predictive AUC: LogisticRegression | AUC=0.61941±0.00000
2) Fastest inference: LogisticRegression | 0.007285 ms/sample
3) Smallest model size: LogisticRegression | 0.011 MB
4) Most stable across seeds: LogisticRegression | AUC std=0.00000
5) Recommendation hint:
   - 若追求极致性能且可接受较高训练/预处理复杂度：优先考虑最优模型（可能是深度模型或 GBDT）。
   - 若强调部署效率、稳定性与维护成本：优先考虑 CatBoost / LightGBM / Logistic 等工程成熟方案。
   - 深度表格模型在复杂类别交互、充足样本、可投入调参资源时更可能体现优势。


## 5) 运行建议

- 先执行到第 2 阶段，确认数据和切分无误。
- 第 3 阶段训练耗时较长（7 个模型 × 3 seeds × 50 trials），建议分模型分批运行。
- 若时间受限，可先把 `N_TRIALS` 临时调小（如 10）做冒烟实验，流程正确后再恢复 50。
- 所有关键输出将写入 `artifacts/`：包括 split 索引、汇总表、测试集预测。